# 06 · FastAPI Serving Layer

This notebook walks through every file in the `api/` directory and explains **why** each design decision was made.  
By the end you will understand how the trained `xgb_default.pkl` model is exposed as a production-grade REST service with:

| Concern | Solution |
|---|---|
| Model serving | **FastAPI** + `joblib` |
| Persistence / audit log | **SQLite** via **SQLAlchemy** ORM |
| Input / output contracts | **Pydantic v2** schemas |
| Error contract | Structured `APIError` responses with named codes |
| Rate limiting | **slowapi** (Starlette middleware, token-bucket per IP) |
| Browser front-end | **HTMX** + Jinja2 templates (no JavaScript framework needed) |

---

## Directory layout

```
api/
├── __init__.py
├── database.py          ← SQLAlchemy engine & session factory
├── models.py            ← ORM table: PredictionLog
├── schemas.py           ← Pydantic I/O schemas + ERROR_CODES contract
├── limiter.py           ← slowapi Limiter singleton
├── main.py              ← FastAPI app, startup, all routes
└── templates/
    ├── index.html           ← HTMX front-end
    ├── result_fragment.html ← HTML partial: single decision card
    └── history_fragment.html← HTML partial: history <tbody> rows
```

---
## Step 1 · Database layer — `api/database.py`

We store every prediction in a local **SQLite** file at `data/predictions.db`.  
This gives us a free audit trail with zero external infrastructure.

### Why SQLite?
- Zero configuration, single file, ships with Python.
- Perfectly adequate for a single-process API server.
- Easy to swap for Postgres later by changing `SQLALCHEMY_DATABASE_URL`.

### Key SQLAlchemy concepts used

| Object | Role |
|---|---|
| `create_engine` | Opens (or creates) the database connection pool |
| `SessionLocal` | A factory that produces one DB session per request |
| `Base` | All ORM model classes inherit from this so SQLAlchemy knows about them |
| `get_db` | FastAPI **dependency** — yields a session, always closes it |

> **`check_same_thread=False`** is required for SQLite when used inside an async
> framework — multiple coroutines may share the same thread.

In [14]:
# ── api/database.py ──────────────────────────────────────────────────────────

from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker
from pathlib import Path

# Resolve path relative to the api/ directory so the DB always lands in data/
DB_PATH = Path("../data/predictions.db").resolve()
SQLALCHEMY_DATABASE_URL = f"sqlite:///{DB_PATH}"

engine = create_engine(
    SQLALCHEMY_DATABASE_URL,
    connect_args={"check_same_thread": False},  # needed for SQLite + async
)

# Session factory — autocommit=False means we control transactions manually
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

# All ORM models must inherit from Base so SQLAlchemy tracks their metadata
Base = declarative_base()


def get_db():
    """FastAPI dependency: yields one DB session per request, always closes it.

    Usage in a route:
        async def my_route(db: Session = Depends(get_db)): ...
    """
    db = SessionLocal()
    try:
        yield db        # hand the session to the route handler
    finally:
        db.close()      # always close, even if the route raised an exception

print(f"Database URL: {SQLALCHEMY_DATABASE_URL}")

Database URL: sqlite:///C:\Users\Kevin\OneDrive\Documents\Programming Projects\credit-card-underwritting\data\predictions.db


---
## Step 2 · ORM model — `api/models.py`

An ORM (Object-Relational Mapper) lets us work with database rows as plain Python objects instead of writing raw SQL.

### Table: `prediction_logs`

We store the full feature dictionary as JSON text so the log is self-contained and queryable.  
The three most business-relevant features (FICO, income, DTI) are also broken out into their own columns for fast filtering without JSON parsing.

| Column | Type | Purpose |
|---|---|---|
| `id` | INTEGER PK | Auto-incrementing surrogate key |
| `decision` | TEXT | `APPROVED` or `DECLINED` |
| `probability` | REAL | Model's raw P(approved) |
| `fico_score` | REAL | Denormalized for fast queries |
| `annual_income` | REAL | Denormalized for fast queries |
| `debt_to_income_ratio` | REAL | Denormalized for fast queries |
| `client_ip` | TEXT | Rate-limit auditing |
| `features_json` | TEXT | Full 78-feature input as JSON |
| `created_at` | DATETIME | UTC timestamp |


In [15]:
# ── api/models.py ────────────────────────────────────────────────────────────

from sqlalchemy import Column, DateTime, Float, Integer, String, Text
from datetime import datetime, timezone

# Base is imported from database.py in the real module;
# here we create a local one just for illustration.
from sqlalchemy.orm import declarative_base
Base = declarative_base()


class PredictionLog(Base):
    """One row per scoring request — our immutable audit trail."""

    __tablename__ = "prediction_logs"

    id       = Column(Integer, primary_key=True, index=True)  # auto PK
    decision = Column(String(10), nullable=False)             # APPROVED | DECLINED
    probability = Column(Float, nullable=False)               # P(approved) from model

    # Denormalised key fields — avoids JSON parsing for the history table
    fico_score           = Column(Float, nullable=True)
    annual_income        = Column(Float, nullable=True)
    debt_to_income_ratio = Column(Float, nullable=True)

    client_ip    = Column(String(45), nullable=True)          # IPv4 or IPv6
    features_json = Column(Text, nullable=False)              # full input snapshot
    created_at   = Column(DateTime, default=lambda: datetime.now(timezone.utc))


# Verify the table schema prints correctly
from sqlalchemy import inspect as sa_inspect, create_engine
tmp_engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(bind=tmp_engine)
cols = [(c.name, str(c.type)) for c in PredictionLog.__table__.columns]
print("prediction_logs columns:")
for name, typ in cols:
    print(f"  {name:<28} {typ}")

prediction_logs columns:
  id                           INTEGER
  decision                     VARCHAR(10)
  probability                  FLOAT
  fico_score                   FLOAT
  annual_income                FLOAT
  debt_to_income_ratio         FLOAT
  client_ip                    VARCHAR(45)
  features_json                TEXT
  created_at                   DATETIME


---
## Step 3 · Pydantic schemas & error contract — `api/schemas.py`

**Pydantic** validates every request body and serialises every response.  
FastAPI uses these schemas to auto-generate the OpenAPI spec and Swagger UI.

### Error contract

All errors follow the same JSON envelope:

```json
{
  "code":    "MISSING_FEATURES",
  "message": "77 required feature(s) missing.",
  "detail":  ["education_level", ...]
}
```

Having a fixed set of named error codes (like gRPC status codes or Stripe's error catalog) means API consumers can write `switch` statements rather than parsing human-readable strings.

### Why `dict[str, float]` for features?

Three column names contain characters illegal in Python identifiers:  
`employment_status_Part-Time`, `employment_status_Self-Employed`, `housing_status_Own Outright`.  
Using a plain dict avoids fighting Pydantic's field-alias machinery for these edge cases.

In [16]:
# ── api/schemas.py ───────────────────────────────────────────────────────────

from pydantic import BaseModel, field_validator
from typing import Any
from datetime import datetime

# ── Error contract ────────────────────────────────────────────────────────────
# A dictionary that documents every possible error code this API can return.
# Served as-is from GET /error-codes so consumers can discover the contract.

ERROR_CODES = {
    "MISSING_FEATURES":    {"status": 400, "description": "One or more required features are absent."},
    "INVALID_FEATURE_VALUE": {"status": 400, "description": "A feature value is non-numeric or out of range."},
    "PREDICTION_FAILED":   {"status": 500, "description": "The model raised an exception during inference."},
    "RATE_LIMIT_EXCEEDED": {"status": 429, "description": "Too many requests. Limit: 10 predictions/minute/IP."},
    "VALIDATION_ERROR":    {"status": 422, "description": "Pydantic schema validation failed."},
    "NOT_FOUND":           {"status": 404, "description": "The requested resource does not exist."},
    "INTERNAL_ERROR":      {"status": 500, "description": "An unexpected server-side error."},
}


# ── Shared error envelope ─────────────────────────────────────────────────────

class APIError(BaseModel):
    code: str       # machine-readable key from ERROR_CODES
    message: str    # human-readable summary
    detail: Any = None  # optional structured context (e.g. list of missing fields)


# ── Prediction request ────────────────────────────────────────────────────────

class PredictionRequest(BaseModel):
    """Wrapper around the 78-feature dict expected by xgb_default.pkl."""

    features: dict[str, float]

    @field_validator("features")
    @classmethod
    def check_numeric(cls, v: dict) -> dict:
        """Pydantic will have already coerced values to float;
        this validator rejects any key whose value cannot be cast."""
        for key, val in v.items():
            if not isinstance(val, (int, float)):
                raise ValueError(
                    f"Feature '{key}' must be numeric, got {type(val).__name__}."
                )
        return v


# ── Prediction response ───────────────────────────────────────────────────────

class PredictionResponse(BaseModel):
    id: int
    decision: str          # "APPROVED" | "DECLINED"
    probability: float     # P(approved) — full precision
    fico_score: float | None
    annual_income: float | None
    debt_to_income_ratio: float | None
    created_at: datetime

    model_config = {"from_attributes": True}  # lets Pydantic read ORM row attributes


# ── Lightweight summary for the list endpoint ─────────────────────────────────

class PredictionSummary(BaseModel):
    id: int
    decision: str
    probability: float
    fico_score: float | None
    annual_income: float | None
    created_at: datetime

    model_config = {"from_attributes": True}


class HealthResponse(BaseModel):
    status: str
    model: str
    feature_count: int
    total_predictions: int


# ── Quick smoke-test ──────────────────────────────────────────────────────────
req = PredictionRequest(features={"fico_score": 700.0, "annual_income": 55000.0})
print("PredictionRequest parsed OK:", req.features)
print("Error codes defined:", list(ERROR_CODES.keys()))

PredictionRequest parsed OK: {'fico_score': 700.0, 'annual_income': 55000.0}
Error codes defined: ['MISSING_FEATURES', 'INVALID_FEATURE_VALUE', 'PREDICTION_FAILED', 'RATE_LIMIT_EXCEEDED', 'VALIDATION_ERROR', 'NOT_FOUND', 'INTERNAL_ERROR']


---
## Step 4 · Rate limiter — `api/limiter.py`

**slowapi** wraps the popular `limits` library and integrates with Starlette/FastAPI.

### How it works

1. `key_func=get_remote_address` — the bucket key is the caller's IP address.  
2. Each `@limiter.limit("N/period")` decorator on a route applies an additional per-route cap on top of the global default.  
3. When the limit is exceeded, slowapi raises `RateLimitExceeded`, which we register a handler for in `main.py`.

### Limit strategy chosen

| Route | Limit | Rationale |
|---|---|---|
| `POST /predict` | **10/min** | Model inference is CPU-bound; prevents DoS |
| `GET /predictions` | **30/min** | Read-only, cheaper, but still needs a guardrail |
| Global default | 60/min | Catch-all for any other route |

In [17]:
# ── api/limiter.py ───────────────────────────────────────────────────────────

from slowapi import Limiter
from slowapi.util import get_remote_address

# A single Limiter instance is created here and imported everywhere.
# Keeping it as a singleton prevents multiple in-memory stores from diverging.

limiter = Limiter(
    key_func=get_remote_address,    # bucket key = caller IP
    default_limits=["60/minute"],   # global cap on every route
)

# In main.py we attach it like this:
#   app.state.limiter = limiter
#   app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)
#
# Then per-route:
#   @limiter.limit("10/minute")
#   async def predict(request: Request, ...):
#       ...

print("Limiter created. Default limits:", limiter._default_limits)

Limiter created. Default limits: [<slowapi.wrappers.LimitGroup object at 0x0000026F346BF650>]


---
## Step 5 · Loading the model

The trained `xgb_default.pkl` is loaded **once at startup** into a module-level variable.  
Loading it inside a route function would re-deserialise the 741 KB file on every request — a ~50ms penalty per call.

### Feature names

XGBoost stores the training column names inside the booster object.  
We extract them with `model.get_booster().feature_names` and use the list to:
- Validate incoming requests (detect missing features)
- Reorder the input DataFrame to guarantee column order matches training

In [18]:
import joblib
import pandas as pd
from pathlib import Path

MODEL_PATH = Path("../models/xgb_default.pkl").resolve()

# joblib.load deserialises the pickled XGBoost classifier
model = joblib.load(MODEL_PATH)

# XGBoost stores the exact training column names inside the booster object.
# This is the ground truth for what the API must receive.
FEATURE_NAMES: list[str] = model.get_booster().feature_names

print(f"Model type   : {type(model).__name__}")
print(f"Feature count: {len(FEATURE_NAMES)}")
print(f"First 5 features: {FEATURE_NAMES[:5]}")
print(f"Last  5 features: {FEATURE_NAMES[-5:]}")

# ── Inference helper ──────────────────────────────────────────────────────────

def score_application(features: dict[str, float]) -> tuple[str, float]:
    """Run model inference on a single application.

    Args:
        features: dict mapping feature name → float value

    Returns:
        (decision, probability) where decision is 'APPROVED' | 'DECLINED'
        and probability is P(approved) in [0, 1].

    Raises:
        ValueError: if any of the 78 required features are missing.
    """
    # 1. Check all required features are present
    missing = [f for f in FEATURE_NAMES if f not in features]
    if missing:
        raise ValueError(f"Missing features: {missing}")

    # 2. Build a single-row DataFrame with columns in training order.
    #    Column ordering matters for tree-based models — wrong order silently
    #    swaps feature importance without raising any error.
    X = pd.DataFrame([features])[FEATURE_NAMES]

    # 3. predict_proba returns [[P(class=0), P(class=1)]]; we want P(approved)
    proba = float(model.predict_proba(X)[0][1])

    # 4. Apply decision threshold (0.5 = default; can be tuned via ROC analysis)
    decision = "APPROVED" if proba >= 0.5 else "DECLINED"

    return decision, proba

Model type   : XGBClassifier
Feature count: 78
First 5 features: ['education_level', 'years_employed', 'annual_income', 'total_household_income', 'monthly_rent_mortgage']
Last  5 features: ['self_reported_monthly_rent_woe', 'monthly_rent_mortgage_woe', 'savings_coverage_months_woe', 'employment_status_Student_woe', 'employment_status_Homemaker_woe']


---
## Step 6 · FastAPI app setup — `api/main.py` (bootstrap)

FastAPI is built on **Starlette** (ASGI framework) and uses Python's `async/await` throughout.

### App object

```python
app = FastAPI(
    title="Credit Card Underwriting API",
    version="1.0.0",
    responses={400: ..., 422: ..., 429: ..., 500: ...},
)
```

The `responses` dict populates the **global responses** section of the OpenAPI spec — every route automatically inherits these error shapes in Swagger UI.

### Jinja2 templates

```python
templates = Jinja2Templates(directory="api/templates")
```

FastAPI can return either JSON *or* HTML from the same route by checking the `HX-Request` header that HTMX sends automatically.  
This is the "islands" pattern: the API is a proper JSON API, but it can also serve HTML fragments directly to the browser.

In [19]:
# ── api/main.py — bootstrap section ─────────────────────────────────────────

import logging
from pathlib import Path

from fastapi import FastAPI
from fastapi.templating import Jinja2Templates
from slowapi import _rate_limit_exceeded_handler
from slowapi.errors import RateLimitExceeded

# In the real module these come from sibling files:
# from .database import Base, engine
# from .limiter import limiter
# from .schemas import APIError, ERROR_CODES

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

app = FastAPI(
    title="Credit Card Underwriting API",
    version="1.0.0",
    description="XGBoost-powered credit decision engine with SQLite audit log.",
    # Global error response shapes — inherited by every route in Swagger UI
    responses={
        400: {"description": "Missing or invalid features"},
        422: {"description": "Pydantic validation error"},
        429: {"description": "Rate limit exceeded"},
        500: {"description": "Internal server error"},
    },
)

# Attach the rate limiter to the app's state.
# slowapi looks for app.state.limiter when processing each request.
# app.state.limiter = limiter
# app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

# Jinja2 template engine — points at api/templates/
TEMPLATES_DIR = Path("../api/templates").resolve()
templates = Jinja2Templates(directory=TEMPLATES_DIR)

# Create all ORM tables (no-op if they already exist)
# Base.metadata.create_all(bind=engine)

print("App created:", app.title, app.version)

App created: Credit Card Underwriting API 1.0.0


---
## Step 7 · Routes — health check & error-code catalog

### `GET /health`
A lightweight liveness probe. Returns:
- `status`: always `"ok"` if the server is running
- `feature_count`: confirms the correct model is loaded
- `total_predictions`: live count from SQLite

### `GET /error-codes`
Exposes the full `ERROR_CODES` dict as JSON.  
This is the machine-readable API contract — client teams can fetch it to build error-handling logic without reading the docs.

In [20]:
# ── api/main.py — /health and /error-codes routes ───────────────────────────

from fastapi import Depends
from sqlalchemy.orm import Session

# ---------------------------------------------------------------------------
# GET /health
# ---------------------------------------------------------------------------
# @app.get("/health", response_model=HealthResponse, tags=["ops"])
async def health_example(db: Session):   # db: Session = Depends(get_db) in real code
    """Liveness check — confirms the model is loaded and DB is reachable."""
    count = db.query(PredictionLog).count()   # one cheap COUNT(*) query
    return HealthResponse(
        status="ok",
        model="xgb_default",
        feature_count=len(FEATURE_NAMES),
        total_predictions=count,
    )

# ---------------------------------------------------------------------------
# GET /error-codes
# ---------------------------------------------------------------------------
# @app.get("/error-codes", tags=["ops"])
async def error_codes_example():
    """Return the machine-readable error contract for this API.

    Consumers can GET this endpoint to discover all possible error codes
    without reading documentation.
    """
    return ERROR_CODES

# Demonstrate the contract inline:
import json
print(json.dumps(ERROR_CODES, indent=2))

{
  "MISSING_FEATURES": {
    "status": 400,
    "description": "One or more required features are absent."
  },
  "INVALID_FEATURE_VALUE": {
    "status": 400,
    "description": "A feature value is non-numeric or out of range."
  },
  "PREDICTION_FAILED": {
    "status": 500,
    "description": "The model raised an exception during inference."
  },
  "RATE_LIMIT_EXCEEDED": {
    "status": 429,
    "description": "Too many requests. Limit: 10 predictions/minute/IP."
  },
  "VALIDATION_ERROR": {
    "status": 422,
    "description": "Pydantic schema validation failed."
  },
  "NOT_FOUND": {
    "status": 404,
    "description": "The requested resource does not exist."
  },
  "INTERNAL_ERROR": {
    "status": 500,
    "description": "An unexpected server-side error."
  }
}


---
## Step 8 · Core prediction route — `POST /predict`

This is the heart of the API. The route:

1. **Validates** the request body with Pydantic (automatic, raises 422 if malformed)
2. **Checks** all 78 feature names are present (raises `MISSING_FEATURES` 400)
3. **Builds** a correctly-ordered pandas DataFrame
4. **Runs** `model.predict_proba` — the only call to the ML model
5. **Persists** the result to SQLite
6. **Returns** JSON *or* an HTML fragment depending on the caller

### HTMX dual-response pattern

HTMX automatically adds the header `HX-Request: true` to every request it makes.  
The route checks for this header and returns an HTML fragment instead of JSON:

```
Browser (HTMX)  →  POST /predict  HX-Request: true  →  HTML fragment
REST client     →  POST /predict                     →  JSON
```

This means the same endpoint works for both the browser UI and programmatic access.

### Rate limit decorator

```python
@limiter.limit("10/minute")
async def predict(request: Request, ...):
```

slowapi requires `Request` to be the **first** parameter so it can extract the IP and increment the counter.

In [21]:
# ── api/main.py — POST /predict ──────────────────────────────────────────────

import json as json_lib
from datetime import datetime, timezone
from fastapi import HTTPException, Request

# ---------------------------------------------------------------------------
# Helper: write a PredictionLog row to the database
# ---------------------------------------------------------------------------

def log_prediction(
    db: Session,
    features: dict,
    decision: str,
    proba: float,
    client_ip: str,
) -> object:  # returns PredictionLog ORM instance
    """Insert one row into prediction_logs and return the hydrated ORM object."""
    entry = PredictionLog(
        decision=decision,
        probability=proba,
        # Pull the three denormalised fields directly from the feature dict
        fico_score=features.get("fico_score"),
        annual_income=features.get("annual_income"),
        debt_to_income_ratio=features.get("debt_to_income_ratio"),
        client_ip=client_ip,
        features_json=json_lib.dumps(features),   # full snapshot for auditability
        created_at=datetime.now(timezone.utc),
    )
    db.add(entry)
    db.commit()       # flush to disk and assign the auto-increment id
    db.refresh(entry) # re-read the row so entry.id is populated
    return entry


# ---------------------------------------------------------------------------
# Route: POST /predict
# ---------------------------------------------------------------------------
#
# The real route in main.py is decorated with:
#   @app.post("/predict", ...)
#   @limiter.limit("10/minute")
#
# Note: @limiter.limit must come AFTER @app.post (inner decorator runs first).

async def predict_example(
    request: Request,          # required by slowapi to extract the caller IP
    body,                      # PredictionRequest in real code
    db: Session = None,        # Depends(get_db) in real code
):
    """Score a single credit application.

    Returns JSON by default.
    Returns an HTML fragment if the request carries HX-Request: true.
    """
    # -- Step 1: check all 78 features are present ---------------------------
    missing = [f for f in FEATURE_NAMES if f not in body.features]
    if missing:
        raise HTTPException(
            status_code=400,
            detail=APIError(
                code="MISSING_FEATURES",
                message=f"{len(missing)} required feature(s) missing.",
                detail=missing,
            ).model_dump(),
        )

    # -- Step 2: run inference ------------------------------------------------
    try:
        decision, proba = score_application(body.features)
    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=APIError(
                code="PREDICTION_FAILED",
                message="Model inference failed.",
                detail=str(exc),
            ).model_dump(),
        )

    # -- Step 3: persist to SQLite --------------------------------------------
    # (omitted here since db is None in this notebook illustration)
    # entry = log_prediction(db, body.features, decision, proba, request.client.host)

    # -- Step 4: return HTML fragment for HTMX, JSON for REST clients ---------
    if request.headers.get("HX-Request") == "true":
        # Return Jinja2 template rendered as HTML — HTMX injects this into the DOM
        return "<html fragment would be returned here>"

    # Standard JSON response
    return {"decision": decision, "probability": proba}


# -- Illustrate the logic end-to-end (no HTTP layer) -------------------------
sample = {k: 0.5 for k in FEATURE_NAMES}
sample["fico_score"] = 750
sample["debt_to_income_ratio"] = 0.25
decision, proba = score_application(sample)
print(f"Decision: {decision}  |  P(approved): {proba:.4f}")

Decision: DECLINED  |  P(approved): 0.3796


---
## Step 9 · Audit history routes

### `GET /predictions` — paginated list

- Accepts `limit` (max 100) and `offset` query params for pagination.
- Returns rows sorted most-recent-first.
- Supports the same HTMX dual-response pattern: returns an HTML `<tbody>` fragment for the browser's live history table.
- The HTMX page polls this endpoint every 5 seconds using `hx-trigger="every 5s"`.

### `GET /predictions/{id}` — single record

- Returns the full `PredictionResponse` including all denormalised fields.
- Returns `404 NOT_FOUND` if the id does not exist.

In [22]:
# ── api/main.py — audit history routes ──────────────────────────────────────

# ---------------------------------------------------------------------------
# GET /predictions  (rate-limited to 30/min)
# ---------------------------------------------------------------------------

# @app.get("/predictions", response_model=list[PredictionSummary], tags=["audit"])
# @limiter.limit("30/minute")
async def list_predictions_example(
    request: Request,
    limit: int = 20,    # how many rows to return (capped at 100)
    offset: int = 0,    # skip the first N rows (for pagination)
    db: Session = None,
):
    """Return prediction history, most recent first.

    The HTMX front-end polls this endpoint every 5 seconds and replaces the
    history table's <tbody> with the returned HTML fragment.
    """
    rows = (
        db.query(PredictionLog)
        .order_by(PredictionLog.created_at.desc())
        .offset(offset)
        .limit(min(limit, 100))   # hard cap prevents accidental large fetches
        .all()
    )

    # HTMX gets an HTML <tbody> fragment; REST clients get JSON
    if request.headers.get("HX-Request") == "true":
        return "<HTML fragment: <tr> rows for each prediction>"

    return rows   # Pydantic serialises ORM objects via from_attributes=True


# ---------------------------------------------------------------------------
# GET /predictions/{id}
# ---------------------------------------------------------------------------

# @app.get("/predictions/{prediction_id}", response_model=PredictionResponse, tags=["audit"])
async def get_prediction_example(prediction_id: int, db: Session = None):
    """Fetch a single prediction record by id."""
    row = (
        db.query(PredictionLog)
        .filter(PredictionLog.id == prediction_id)
        .first()
    )

    if not row:
        # Raise a structured 404 using our error contract
        raise HTTPException(
            status_code=404,
            detail=APIError(
                code="NOT_FOUND",
                message=f"Prediction {prediction_id} not found.",
            ).model_dump(),
        )

    return row

print("History route logic illustrated (no live DB in notebook context)")

History route logic illustrated (no live DB in notebook context)


---
## Step 10 · HTMX front-end — `api/templates/index.html`

### What is HTMX?

HTMX is a ~14 KB library that lets HTML elements make HTTP requests and swap parts of the DOM — without writing JavaScript.  
The key attributes used here:

| Attribute | Effect |
|---|---|
| `hx-post="/predict"` | POST to /predict on form submit |
| `hx-target="#result-panel"` | Replace `#result-panel`'s content with the response |
| `hx-swap="innerHTML"` | Replace the inner HTML (not the element itself) |
| `hx-indicator="#spinner"` | Show `#spinner` while the request is in flight |
| `hx-get="/predictions"` | GET /predictions … |
| `hx-trigger="every 5s"` | … every 5 seconds (live history polling) |

### Why not React/Vue?

For an internal tooling page with one form and one table, shipping a full SPA framework would be  
massive over-engineering. HTMX gives us interactivity with zero build step and zero npm.

### Form → JSON conversion

HTMX does not natively serialize forms as `application/json` (it uses `multipart/form-data` or `x-www-form-urlencoded`).  
A small `preparePayload()` function intercepts the submit event, builds the `{features: {...}}` JSON body, and calls `fetch` directly.

In [23]:
# The critical pieces of the HTMX template, shown as annotated Python strings.
# The full file lives at api/templates/index.html

htmx_snippets = {

    # 1. Form declaration — HTMX attributes define the entire request lifecycle
    "form_tag": """
    <form id="predict-form"
          hx-post="/predict"           <!-- POST to this endpoint -->
          hx-target="#result-panel"   <!-- put the response HTML here -->
          hx-swap="innerHTML"          <!-- replace inner content only -->
          hx-indicator="#spinner"      <!-- show spinner during request -->
          hx-on::before-request="clearError()"
          hx-on::response-error="showError(event)"
          onsubmit="return preparePayload(event)">
    """,

    # 2. History table — auto-polls the server every 5 seconds
    "history_tbody": """
    <tbody id="history-body"
           hx-get="/predictions"       <!-- GET this endpoint -->
           hx-trigger="load, every 5s" <!-- on page load + every 5 seconds -->
           hx-target="#history-body"   <!-- replace itself -->
           hx-swap="innerHTML">
    """,

    # 3. JavaScript: form serialisation to JSON
    "payload_builder": """
    function preparePayload(evt) {
        evt.preventDefault();
        const features = {};
        // Collect every named input and convert to float
        document.querySelectorAll('#predict-form input[name]').forEach(input => {
            const v = input.value.trim();
            if (v !== '') features[input.name] = parseFloat(v);
        });
        fetch('/predict', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json',
                'HX-Request': 'true',   // tell FastAPI to return HTML
            },
            body: JSON.stringify({ features }),
        })
        .then(async res => {
            const text = await res.text();
            if (!res.ok) { showError(...); }
            else {
                document.getElementById('result-panel').innerHTML = text;
                htmx.trigger('#history-body', 'load');  // force history refresh
            }
        });
        return false;
    }
    """,
}

for name, snippet in htmx_snippets.items():
    print(f"── {name} ──")
    print(snippet)

── form_tag ──

    <form id="predict-form"
          hx-post="/predict"           <!-- POST to this endpoint -->
          hx-target="#result-panel"   <!-- put the response HTML here -->
          hx-swap="innerHTML"          <!-- replace inner content only -->
          hx-indicator="#spinner"      <!-- show spinner during request -->
          hx-on::before-request="clearError()"
          hx-on::response-error="showError(event)"
          onsubmit="return preparePayload(event)">
    
── history_tbody ──

    <tbody id="history-body"
           hx-get="/predictions"       <!-- GET this endpoint -->
           hx-trigger="load, every 5s" <!-- on page load + every 5 seconds -->
           hx-target="#history-body"   <!-- replace itself -->
           hx-swap="innerHTML">
    
── payload_builder ──

    function preparePayload(evt) {
        evt.preventDefault();
        const features = {};
        // Collect every named input and convert to float
        document.querySelectorAll('#p

---
## Step 11 · HTML response fragments

When HTMX submits a request, the server returns a **partial HTML fragment** — not a full page.  
HTMX injects this fragment directly into the target element.

### `result_fragment.html`
Rendered by the `POST /predict` route when `HX-Request: true`.  
Receives the `entry` (ORM object) as template context and produces a styled decision card.

### `history_fragment.html`
Rendered by `GET /predictions` when `HX-Request: true`.  
Produces `<tr>` rows that replace the history table's `<tbody>` every 5 seconds.

In [24]:
# The Jinja2 template syntax used in result_fragment.html

result_fragment = """
{# approved is True if the model said APPROVED #}
{% set approved = entry.decision == 'APPROVED' %}

<div class="result-card {{ 'approved' if approved else 'declined' }}">
  <div class="result-icon">{{ '✓' if approved else '✗' }}</div>
  <div class="result-body">
    <h2>{{ entry.decision }}</h2>

    {# Jinja2 format filter formats floats inline #}
    <p>Approval probability: <strong>{{ "%.1f"|format(entry.probability * 100) }}%</strong></p>

    <div class="result-meta">
      <span>FICO: <strong>{{ "%.0f"|format(entry.fico_score) if entry.fico_score else '—' }}</strong></span>
      <span>Income: <strong>${{ "{:,.0f}".format(entry.annual_income) if entry.annual_income else '—' }}</strong></span>
      <span>Record #{{ entry.id }}</span>
    </div>
  </div>
</div>
"""

history_fragment = """
{% if rows %}
  {% for row in rows %}
  <tr class="{{ 'row-approved' if row.decision == 'APPROVED' else 'row-declined' }}">
    <td>#{{ row.id }}</td>
    <td><span class="badge badge-{{ row.decision | lower }}">{{ row.decision }}</span></td>
    <td>{{ "%.1f"|format(row.probability * 100) }}%</td>
    <td>{{ "%.0f"|format(row.fico_score) if row.fico_score else '—' }}</td>
    <td>${{ "{:,.0f}".format(row.annual_income) if row.annual_income else '—' }}</td>
    <td>{{ row.created_at.strftime("%H:%M:%S") }}</td>
  </tr>
  {% endfor %}
{% else %}
  <tr><td colspan="6">No predictions yet.</td></tr>
{% endif %}
"""

print("result_fragment.html:\n", result_fragment[:300], "...")
print("\nhistory_fragment.html:\n", history_fragment[:300], "...")

result_fragment.html:
 
{# approved is True if the model said APPROVED #}
{% set approved = entry.decision == 'APPROVED' %}

<div class="result-card {{ 'approved' if approved else 'declined' }}">
  <div class="result-icon">{{ '✓' if approved else '✗' }}</div>
  <div class="result-body">
    <h2>{{ entry.decision }}</h2>

 ...

history_fragment.html:
 
{% if rows %}
  {% for row in rows %}
  <tr class="{{ 'row-approved' if row.decision == 'APPROVED' else 'row-declined' }}">
    <td>#{{ row.id }}</td>
    <td><span class="badge badge-{{ row.decision | lower }}">{{ row.decision }}</span></td>
    <td>{{ "%.1f"|format(row.probability * 100) }}%</td> ...


---
## Step 12 · Running & testing the API

### Start the server

```bash
source .cc_venv/bin/activate
uvicorn api.main:app --reload --port 8000
```

- `--reload` watches source files and hot-restarts on changes (dev only).
- Open **http://localhost:8000** for the HTMX UI.
- Open **http://localhost:8000/docs** for Swagger UI.

### Test programmatically with `httpx`

The cell below uses **httpx** with FastAPI's `ASGITransport` to run requests in-process — no real server needed.

In [25]:
import sys
sys.path.insert(0, '..')

import httpx
from api.main import app   # import the real FastAPI app

# Example application — raw human-readable inputs (API derives all 78 features)
EXAMPLE_APPLICATION = {
    "education_level":               "Bachelor Degree",
    "employment_status":             "Full-Time",
    "housing_status":                "Rent",
    "years_employed":                5,
    "recent_employment_change":      False,
    "has_existing_mortgage":         False,
    "annual_income":                 42_280,
    "total_household_income":        42_100,
    "savings_account_balance":       2_986,
    "retirement_account_balance":    10_478,
    "avg_monthly_deposits":          3_750,
    "avg_monthly_withdrawals":       3_650,
    "payroll_direct_deposit_amount": 2_900,
    "total_monthly_expenses":        2_362,
    "monthly_rent_mortgage":         959,
    "self_reported_monthly_rent":    959,
    "fico_score":                    532,
    "credit_utilization_ratio":      0.58,
    "debt_to_income_ratio":          0.67,
    "oldest_account_age_months":     181,
    "num_open_accounts":             6,
    "num_student_loans":             2,
    "student_loan_outstanding_balance": 8_362,
    "mortgage_outstanding_balance":  0,
    "requested_credit_limit":        7_500,
    "predicted_default_probability": 0.43,
    "employment_stability_score":    0.48,
    "income_stability_score":        0.43,
    "financial_health_score":        0.52,
    "combined_risk_score":           336,
}

# Jupyter already runs an async event loop, so we use `await` directly
# instead of asyncio.run() which would raise "cannot be called from a running loop".
# ASGITransport runs the ASGI app in-process — no network or port needed.

transport = httpx.ASGITransport(app=app)

async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:

    # ── Test 1: Health check ──────────────────────────────────────────────────
    r = await client.get("/health")
    assert r.status_code == 200
    print("[PASS] GET /health:", r.json())

    # ── Test 2: Successful prediction ─────────────────────────────────────────
    r = await client.post("/predict", json=EXAMPLE_APPLICATION)
    assert r.status_code == 200, r.text
    result = r.json()
    print(f"[PASS] POST /predict: {result['decision']}  P={result['probability']:.4f}")

    # ── Test 3: Invalid field value → 422 Validation Error ────────────────────
    # The API now uses Literal types, so an invalid employment_status is caught
    # by Pydantic before any feature engineering runs.
    bad = dict(EXAMPLE_APPLICATION, employment_status="InvalidStatus")
    r = await client.post("/predict", json=bad)
    assert r.status_code == 422
    print(f"[PASS] Invalid enum value → 422 Unprocessable Entity")

    # ── Test 4: Non-existent record → 404 NOT_FOUND ───────────────────────────
    r = await client.get("/predictions/999999")
    assert r.status_code == 404
    err = r.json()["detail"]
    assert err["code"] == "NOT_FOUND"
    print(f"[PASS] Missing record → 404 {err['code']}")

    # ── Test 5: Error code catalog ────────────────────────────────────────────
    r = await client.get("/error-codes")
    assert r.status_code == 200
    print(f"[PASS] GET /error-codes: {list(r.json().keys())}")

    # ── Test 6: Prediction history ────────────────────────────────────────────
    r = await client.get("/predictions?limit=5")
    assert r.status_code == 200
    print(f"[PASS] GET /predictions: {len(r.json())} row(s) returned")


INFO:api.main:Model loaded. Features: 78


INFO:httpx:HTTP Request: GET http://test/health "HTTP/1.1 200 OK"


[PASS] GET /health: {'status': 'ok', 'model': 'xgb_default', 'feature_count': 78, 'total_predictions': 43}


INFO:httpx:HTTP Request: POST http://test/predict "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://test/predict "HTTP/1.1 422 Unprocessable Entity"
INFO:httpx:HTTP Request: GET http://test/predictions/999999 "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET http://test/error-codes "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://test/predictions?limit=5 "HTTP/1.1 200 OK"


[PASS] POST /predict: APPROVED  P=0.9997
[PASS] Invalid enum value → 422 Unprocessable Entity
[PASS] Missing record → 404 NOT_FOUND
[PASS] GET /error-codes: ['MISSING_FEATURES', 'INVALID_FEATURE_VALUE', 'PREDICTION_FAILED', 'RATE_LIMIT_EXCEEDED', 'VALIDATION_ERROR', 'NOT_FOUND', 'INTERNAL_ERROR']
[PASS] GET /predictions: 5 row(s) returned


---
## Step 13 · End-to-end: start the live server

Run the cell below to start the server in a background thread, then use `requests` to hit it over a real TCP connection.

In [26]:
import threading, time, requests, socket
import uvicorn
import sys
sys.path.insert(0, '..')
from api.main import app  # import the object directly — avoids string-import path issues

# ── Port guard ────────────────────────────────────────────────────────────────
# If a previous kernel run left a server on port 8000, this cell would crash
# with "Address already in use". Check first and warn clearly.
PORT = 8000

def port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if port_in_use(PORT):
    print(f"⚠️  Port {PORT} is already in use.")
    print("   Run `server.should_exit = True` in a new cell, or restart the kernel, then re-run this cell.")
    server = None
else:
    config = uvicorn.Config(app=app, host="127.0.0.1", port=PORT, log_level="warning")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    while not server.started:
        time.sleep(0.1)

    BASE = f"http://127.0.0.1:{PORT}"

    # ── Health check over real HTTP ───────────────────────────────────────────
    health = requests.get(f"{BASE}/health").json()
    print("Health:", health)

    # ── Score an application over real HTTP ───────────────────────────────────
    EXAMPLE_APPLICATION = {
        "education_level":               "Bachelor Degree",
        "employment_status":             "Full-Time",
        "housing_status":                "Rent",
        "years_employed":                5,
        "recent_employment_change":      False,
        "has_existing_mortgage":         False,
        "annual_income":                 42280,
        "total_household_income":        42100,
        "savings_account_balance":       2986,
        "retirement_account_balance":    10478,
        "avg_monthly_deposits":          3750,
        "avg_monthly_withdrawals":       3650,
        "payroll_direct_deposit_amount": 2900,
        "total_monthly_expenses":        2362,
        "monthly_rent_mortgage":         959,
        "self_reported_monthly_rent":    959,
        "fico_score":                    532,
        "credit_utilization_ratio":      0.58,
        "debt_to_income_ratio":          0.67,
        "oldest_account_age_months":     181,
        "num_open_accounts":             6,
        "num_student_loans":             2,
        "student_loan_outstanding_balance": 8362,
        "mortgage_outstanding_balance":  0,
        "requested_credit_limit":        7500,
        "predicted_default_probability": 0.43,
        "employment_stability_score":    0.48,
        "income_stability_score":        0.43,
        "financial_health_score":        0.52,
        "combined_risk_score":           336,
    }

    resp = requests.post(f"{BASE}/predict", json=EXAMPLE_APPLICATION, timeout=10)
    print(f"Prediction: {resp.status_code}", resp.json())

    print(f"\nBrowser UI : {BASE}/")
    print(f"Swagger UI : {BASE}/docs")
    print(f"\nTo stop    : server.should_exit = True")


Health: {'status': 'ok', 'model': 'xgb_default', 'feature_count': 78, 'total_predictions': 44}
Prediction: 200 {'id': 45, 'decision': 'APPROVED', 'probability': 0.9996564388275146, 'fico_score': 532.0, 'annual_income': 42280.0, 'debt_to_income_ratio': 0.67, 'created_at': '2026-04-02T09:54:42.088748'}

Browser UI : http://127.0.0.1:8000/
Swagger UI : http://127.0.0.1:8000/docs

To stop    : server.should_exit = True


In [27]:
if server is not None:
    server.should_exit = True  # signal the server to shut down after the tests


---
## Summary

The table below maps every design requirement to the solution chosen:

| Requirement | File | Key detail |
|---|---|---|
| Serve `xgb_default.pkl` | `main.py` | `joblib.load` once at startup; `predict_proba` per request |
| Persist every decision | `models.py` + `database.py` | SQLite via SQLAlchemy ORM, `data/predictions.db` |
| Validate 78 features | `schemas.py` | `PredictionRequest` + manual missing-feature check |
| Error code contract | `schemas.py` | `APIError` envelope + `ERROR_CODES` catalog at `GET /error-codes` |
| Rate limiting | `limiter.py` + `main.py` | slowapi, 10 pred/min on `POST /predict`, 30/min on `GET /predictions` |
| Browser front-end | `templates/index.html` | HTMX + Jinja2, tabbed form, live polling every 5 s |
| JSON + HTML from one route | `main.py` | `HX-Request` header switch; Jinja2 fragments for HTMX, Pydantic JSON for REST |

### Next steps

- **Threshold tuning**: The current 0.5 decision threshold is the model default. Notebook `07_model_evaluation.ipynb` should inform the optimal operating point from the ROC curve.
- **Authentication**: Add an API key header check (middleware or dependency) before exposing to external clients.
- **Preprocessing pipeline**: Accept raw application data and run the full preprocessing chain (imputation → encoding → WoE) server-side so callers don't need to pre-compute the 78 features.
- **Docker**: Package with a `Dockerfile` (`FROM python:3.13-slim`, `CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0"]`) for portable deployment.

---
# Update · Feature Engineering Pipeline

## The Problem with the Original Design

The original `/predict` endpoint required callers to supply **all 78 pre-processed features**,
including values that normal users cannot reasonably know or compute:

| Category | Examples | Why users can't provide these |
|---|---|---|
| Log-scaled fields | `annual_income = 10.65` | Raw income is \$42,280 — no one thinks in `log1p` |
| Interaction features | `fico_utilization_score`, `income_dti_capacity` | These are computed from other inputs |
| WoE-encoded fields | `fico_score_woe`, `debt_to_income_ratio_woe` | Requires training-data bin maps to compute |

This meant the browser UI had a **WoE Features** tab with 33 fields no user could fill in,
and the Load Example button just pasted pre-computed numbers from X_train.

## The Solution

We moved all feature engineering **server-side**. The API now accepts ~28 human-readable inputs
and automatically computes the full 78-feature vector before scoring.

```
Before:  User provides 78 preprocessed features  →  model.predict_proba()
After:   User provides 28 raw inputs  →  engineer_features()  →  model.predict_proba()
```

### New file: `api/feature_engineering.py`

This module contains the full pipeline:
1. **Ordinal encoding** — education level (string → 1–7), FICO tier (score → 1–5)
2. **Binary encoding** — yes/no flags → 0/1
3. **One-hot encoding** — employment status and housing status
4. **Log1p transforms** — right-skewed dollar amounts (income, balances, deposits)
5. **Derived features** — disposable income, ratios, interaction terms
6. **WoE encoding** — 33 features encoded using bin maps rebuilt from training data


---
## Step A · Log1p Transforms

Dollar amounts like income and balances are **right-skewed** — a few applicants
have very high values that would dominate distance-based calculations.
`np.log1p(x)` (= ln(1 + x)) compresses large values while preserving the ordering.

The `+1` inside the log means `log1p(0) = 0`, so zero balances stay zero.

```
Raw annual_income = $42,280  →  log1p(42280) = 10.65  ✓ (matches X_train)
Raw annual_income =      $0  →  log1p(0)     =  0.00  ✓
```

**Columns that get log1p'd (verified against X_train vs raw data):**
- `annual_income`, `retirement_account_balance`
- `avg_monthly_deposits`, `avg_monthly_withdrawals`, `payroll_direct_deposit_amount`
- `mortgage_outstanding_balance`, `student_loan_outstanding_balance`
- `income_to_requested_limit_ratio` (the *ratio* is log1p'd, not the components separately)

**Not log1p'd** (already on a reasonable scale or can be negative):
`fico_score`, `debt_to_income_ratio`, `total_monthly_expenses`, `monthly_disposable_income`, etc.

In [28]:
import numpy as np

# Verify: raw annual_income $42,280 → matches the value seen in X_train
raw_income = 42_280
transformed = np.log1p(raw_income)
print(f"log1p({raw_income:,}) = {transformed:.6f}")
print(f"X_train stored:        10.652093  ← same value")

# Why not just log()?
print(f"\nlog(0)    = {float('-inf')}  ← crashes on zero balances")
print(f"log1p(0)  = {np.log1p(0)}     ← safe")

log1p(42,280) = 10.652093
X_train stored:        10.652093  ← same value

log(0)    = -inf  ← crashes on zero balances
log1p(0)  = 0.0     ← safe


---
## Step B · Derived Features

These features are **computed from the raw inputs** — users never need to enter them.

| Feature | Formula | Interpretation |
|---|---|---|
| `monthly_disposable_income` | `annual_income / 12 − total_monthly_expenses` | Cash left over each month |
| `disposable_income_ratio` | `monthly_disposable / (annual_income / 12)` | Fraction of income available |
| `fico_score_tier` | FICO → 1–5 bucket | Very Poor/Fair/Good/Very Good/Exceptional |
| `income_to_requested_limit_ratio` | `log1p(annual_income / requested_limit)` | Income relative to credit ask |
| `fico_utilization_score` | `fico_score × (1 − credit_utilization_ratio)` | High FICO + low utilization = good |
| `income_dti_capacity` | `annual_income × (1 − dti)` | Income available after existing debt |
| `fico_default_alignment` | `fico_score × (1 − predicted_default_prob)` | FICO and default model agree |
| `health_risk_composite` | `financial_health_score × combined_risk_score` | Two risk models compounded |
| `savings_coverage_months` | `log1p(savings_balance) / (total_expenses + 1)` | Months of expenses savings can cover |
| `debt_burden_ratio` | `debt_to_income_ratio` | Same value, kept as a separate signal |
| `credit_depth` | `num_open_accounts / (oldest_account_age_months / 12 + 1)` | Account density per year of history |
| `income_cushion` | `monthly_disposable / (log1p(requested_limit) + 1)` | Spare income relative to credit ask |

In [29]:
import numpy as np
import pandas as pd

# Verify derived features match X_train for the example applicant
# (same applicant used in the original notebook example)
annual_income_raw       = 42_280.0
total_monthly_expenses  = 2_362.0
fico_score              = 532.0
credit_utilization_ratio = 0.5835
debt_to_income_ratio    = 0.6704
predicted_default_prob  = 0.4337
financial_health_score  = 0.5156
combined_risk_score     = 336.0
savings_balance         = 2_985.56
num_open_accounts       = 6.0
oldest_account_age_months = 181.0
requested_credit_limit  = 7_500.0

monthly_income              = annual_income_raw / 12
monthly_disposable_income   = monthly_income - total_monthly_expenses
disposable_income_ratio     = monthly_disposable_income / monthly_income
fico_utilization_score      = fico_score * (1 - min(credit_utilization_ratio, 1))
income_dti_capacity         = annual_income_raw * (1 - min(debt_to_income_ratio, 1))
fico_default_alignment      = fico_score * (1 - predicted_default_prob)
health_risk_composite       = financial_health_score * combined_risk_score
savings_coverage_months     = np.log1p(savings_balance) / (total_monthly_expenses + 1)
credit_depth                = num_open_accounts / (oldest_account_age_months / 12 + 1)
income_cushion              = monthly_disposable_income / (np.log1p(requested_credit_limit) + 1)

derived = {
    'monthly_disposable_income':  monthly_disposable_income,
    'disposable_income_ratio':     disposable_income_ratio,
    'fico_utilization_score':      fico_utilization_score,
    'income_dti_capacity':         income_dti_capacity,
    'fico_default_alignment':      fico_default_alignment,
    'health_risk_composite':       health_risk_composite,
    'savings_coverage_months':     savings_coverage_months,
    'credit_depth':                credit_depth,
    'income_cushion':              income_cushion,
}

# Compare to what X_train stores for the same applicant
x_train_values = {
    'monthly_disposable_income':  1161.33,
    'disposable_income_ratio':     0.3296,
    'fico_utilization_score':      221.578,
    'income_dti_capacity':         13935.488,
    'fico_default_alignment':      301.2716,
    'health_risk_composite':       173.2416,
    'savings_coverage_months':     0.003386,
    'credit_depth':                0.37306,
    'income_cushion':              117.0366,
}

print(f"{'Feature':<35} {'Computed':>12} {'X_train':>12} {'Match':>8}")
print('-' * 72)
for k in derived:
    c = derived[k]
    x = x_train_values[k]
    ok = '✓' if abs(c - x) < 0.01 else '✗'
    print(f"{k:<35} {c:>12.4f} {x:>12.4f} {ok:>8}")

Feature                                 Computed      X_train    Match
------------------------------------------------------------------------
monthly_disposable_income              1161.3333    1161.3300        ✓
disposable_income_ratio                   0.3296       0.3296        ✓
fico_utilization_score                  221.5780     221.5780        ✓
income_dti_capacity                   13935.4880   13935.4880        ✓
fico_default_alignment                  301.2716     301.2716        ✓
health_risk_composite                   173.2416     173.2416        ✓
savings_coverage_months                   0.0034       0.0034        ✓
credit_depth                              0.3731       0.3731        ✓
income_cushion                          117.0370     117.0366        ✓


---
## Step C · Weight-of-Evidence (WoE) Encoding

### What is WoE?

WoE converts a feature into **log-odds units**, making each feature directly comparable
regardless of its original scale.

For a given bin of a feature:
```
WoE = ln( % of approvals in bin  /  % of declines in bin )

WoE > 0  →  this bin has more approvals than average  (good signal)
WoE < 0  →  this bin has more declines than average   (bad signal)
WoE = 0  →  neutral
```

### Why the WoE maps weren't saved originally

The WoE maps were computed in notebook `02_data_preprocessing.ipynb` and stored in a
Python dict in memory. When the notebook session ended, the maps were lost — only the
already-encoded columns were saved to `cc_underwriting_preprocessed.csv`.

### How we rebuilt them

Since `X_train.csv` contains **both** the base features and their `_woe` counterparts,
we can reconstruct the bin→WoE mapping directly:

1. For each WoE feature, bin the base column using `pd.qcut` (10 quantile bins)
2. For each bin, take the mean WoE value from the `_woe` column
3. Save all bin edges + WoE values to `models/woe_maps.json`

At inference time, a new value is assigned to a bin and looks up the saved WoE.

In [30]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

X_train = pd.read_csv('../data/processed/X_train.csv')

# Illustrate how the WoE map for fico_score is built
base_col = 'fico_score'
woe_col  = 'fico_score_woe'

# 1. Bin fico_score into 10 quantiles, get the corresponding WoE values, and average them per bin
binned, bin_edges = pd.qcut(X_train[base_col], q=10, duplicates='drop', retbins=True) # bins are automatically labeled with their interval ranges
tmp = pd.DataFrame({'bin': binned, 'woe': X_train[woe_col]})
bin_woe = tmp.groupby('bin')['woe'].mean().round(4)

print(f"WoE map for '{base_col}' (10 quantile bins):")
print(f"{'Bin range':<25} {'WoE':>8}  {'Interpretation':<30}")
print('-' * 67)
for interval, woe in bin_woe.items():
    interp = 'Approvals > Declines' if woe > 0 else ('Declines > Approvals' if woe < 0 else 'Neutral')
    print(f"  {str(interval):<23} {woe:>8.4f}  {interp}")

# Verify the full woe_maps.json was built and saved
woe_maps_path = Path('../models/woe_maps.json')
with open(woe_maps_path) as f:
    woe_maps = json.load(f)
print(f"\nwoe_maps.json: {len(woe_maps)} features saved ✓")

WoE map for 'fico_score' (10 quantile bins):
Bin range                      WoE  Interpretation                
-------------------------------------------------------------------
  (314.999, 453.0]         -4.3473  Declines > Approvals
  (453.0, 477.0]           -2.4687  Declines > Approvals
  (477.0, 494.0]           -1.3612  Declines > Approvals
  (494.0, 510.0]           -0.5611  Declines > Approvals
  (510.0, 527.0]            0.2729  Approvals > Declines
  (527.0, 544.0]            1.2742  Approvals > Declines
  (544.0, 564.0]            2.4895  Approvals > Declines
  (564.0, 590.0]            4.5999  Approvals > Declines
  (590.0, 627.0]            7.1956  Approvals > Declines
  (627.0, 842.0]            7.3303  Approvals > Declines

woe_maps.json: 261 features saved ✓


---
## Step D · Updated Schema — `ApplicationRequest`

The original `PredictionRequest` accepted a raw `dict[str, float]` with all 78 features.
The new `ApplicationRequest` accepts typed, named fields that a real applicant or
loan officer would know.

**Before:** `{"features": {"annual_income": 10.65, "fico_utilization_score": 221.58, ...}}` ← 78 fields, pre-processed

**After:** `{"annual_income": 42280, "fico_score": 532, "employment_status": "Full-Time", ...}` ← 28 raw fields

Using `Literal` types for `employment_status`, `housing_status`, and `education_level`
means FastAPI auto-generates dropdown options in Swagger UI and rejects invalid values
with a clear 422 error before any code runs.

In [31]:
import sys
sys.path.insert(0, '..')

from api.schemas import ApplicationRequest

# Show the fields the user now provides
fields = ApplicationRequest.model_fields
print(f"ApplicationRequest has {len(fields)} fields (vs 78 in the old PredictionRequest)")
print()
for name, info in fields.items():
    required = '(required)' if info.is_required() else f'(default={info.default!r})'
    print(f"  {name:<45} {required}")

ApplicationRequest has 30 fields (vs 78 in the old PredictionRequest)

  education_level                               (required)
  employment_status                             (required)
  housing_status                                (required)
  years_employed                                (required)
  recent_employment_change                      (default=False)
  annual_income                                 (required)
  total_household_income                        (required)
  monthly_rent_mortgage                         (required)
  self_reported_monthly_rent                    (required)
  total_monthly_expenses                        (required)
  fico_score                                    (required)
  credit_utilization_ratio                      (required)
  oldest_account_age_months                     (required)
  num_open_accounts                             (required)
  debt_to_income_ratio                          (required)
  num_student_loans                    

---
## Step E · `api/feature_engineering.py` — The Pipeline

The `engineer_features(inp)` function is the core of the update.
It takes the raw `ApplicationRequest` dict and returns the 78-feature dict
the model expects, in the correct order.

**Execution order matters** — WoE encoding must happen last because it reads
the already-computed derived features (e.g., `health_risk_composite_woe` needs
`health_risk_composite` to exist first).

```
Raw inputs
    │
    ├─► Log1p transforms        (annual_income, balances, deposits, ...)
    ├─► Ordinal encoding        (education_level string → int, fico_score → tier)
    ├─► One-hot encoding        (employment_status, housing_status → binary columns)
    ├─► Derived features        (disposable income, ratios, interaction terms)
    └─► WoE encoding            (33 features → _woe variants via woe_maps.json)
                                        │
                                78-feature dict → xgb_default.pkl
```

In [32]:
import sys, joblib
import pandas as pd
sys.path.insert(0, '..')

from api.feature_engineering import engineer_features

# Simplified input — what a user actually fills in
raw_input = {
    'education_level':              'Bachelor Degree',
    'employment_status':            'Full-Time',
    'housing_status':               'Rent',
    'years_employed':               5,
    'recent_employment_change':     False,
    'has_existing_mortgage':        False,
    'annual_income':                42_280,      # raw $
    'total_household_income':       42_100,
    'savings_account_balance':      2_986,
    'retirement_account_balance':   10_478,
    'avg_monthly_deposits':         3_750,
    'avg_monthly_withdrawals':      3_650,
    'payroll_direct_deposit_amount': 2_900,
    'total_monthly_expenses':       2_362,
    'monthly_rent_mortgage':        959,
    'self_reported_monthly_rent':   959,
    'fico_score':                   532,
    'credit_utilization_ratio':     0.58,
    'debt_to_income_ratio':         0.67,
    'oldest_account_age_months':    181,
    'num_open_accounts':            6,
    'num_student_loans':            2,
    'student_loan_outstanding_balance': 8_362,
    'mortgage_outstanding_balance': 0,
    'requested_credit_limit':       7_500,
    'predicted_default_probability': 0.43,
    'employment_stability_score':   0.48,
    'income_stability_score':       0.43,
    'financial_health_score':       0.52,
    'combined_risk_score':          336,
}

features = engineer_features(raw_input)

print(f"Raw inputs:      {len(raw_input)} fields")
print(f"Model features:  {len(features)} fields")
print()
print("Sample of engineered values:")
for k in ['annual_income', 'fico_score_tier', 'employment_status_Unemployed',
          'housing_status_Rent', 'income_dti_capacity', 'fico_score_woe',
          'health_risk_composite_woe']:
    print(f"  {k:<40} {features[k]:.4f}")

Raw inputs:      30 fields
Model features:  78 fields

Sample of engineered values:
  annual_income                            10.6521
  fico_score_tier                          1.0000
  employment_status_Unemployed             0.0000
  housing_status_Rent                      1.0000
  income_dti_capacity                      13952.4000
  fico_score_woe                           1.2742
  health_risk_composite_woe                4.5250


---
## Step F · Updated `POST /predict` Route

The route now calls `engineer_features()` between validation and scoring:

```python
# Before
async def predict(request, body: PredictionRequest, db):
    decision, proba = _score(body.features)          # caller must pre-compute 78 features

# After
async def predict(request, body: ApplicationRequest, db):
    raw     = body.model_dump()                      # 28 raw fields
    features = engineer_features(raw)               # → 78 model-ready features
    decision, proba = _score(features)
```

The `_score()` helper and the rest of the route (HTMX detection, DB logging,
response format) are unchanged.

---
## Step G · End-to-End Test with Simple Inputs

In [33]:
import sys, httpx
sys.path.insert(0, '..')
from api.main import app

APPROVED_APPLICANT = {
    'education_level':              'Bachelor Degree',
    'employment_status':            'Full-Time',
    'housing_status':               'Rent',
    'years_employed':               5,
    'recent_employment_change':     False,
    'has_existing_mortgage':        False,
    'annual_income':                75_000,
    'total_household_income':       90_000,
    'savings_account_balance':      15_000,
    'retirement_account_balance':   25_000,
    'avg_monthly_deposits':         6_500,
    'avg_monthly_withdrawals':      5_800,
    'payroll_direct_deposit_amount': 5_500,
    'total_monthly_expenses':       3_000,
    'monthly_rent_mortgage':        1_500,
    'self_reported_monthly_rent':   1_500,
    'fico_score':                   720,
    'credit_utilization_ratio':     0.20,
    'debt_to_income_ratio':         0.30,
    'oldest_account_age_months':    84,
    'num_open_accounts':            5,
    'num_student_loans':            1,
    'student_loan_outstanding_balance': 10_000,
    'mortgage_outstanding_balance': 0,
    'requested_credit_limit':       5_000,
    'predicted_default_probability': 0.10,
    'employment_stability_score':   0.80,
    'income_stability_score':       0.80,
    'financial_health_score':       0.72,
    'combined_risk_score':          280,
}

DECLINED_APPLICANT = {
    'education_level':              'High School Diploma',
    'employment_status':            'Unemployed',
    'housing_status':               'Rent',
    'years_employed':               0,
    'recent_employment_change':     True,
    'has_existing_mortgage':        False,
    'annual_income':                18_000,
    'total_household_income':       18_000,
    'savings_account_balance':      200,
    'retirement_account_balance':   0,
    'avg_monthly_deposits':         1_400,
    'avg_monthly_withdrawals':      1_600,
    'payroll_direct_deposit_amount': 0,
    'total_monthly_expenses':       2_800,
    'monthly_rent_mortgage':        1_200,
    'self_reported_monthly_rent':   1_200,
    'fico_score':                   520,
    'credit_utilization_ratio':     0.95,
    'debt_to_income_ratio':         0.85,
    'oldest_account_age_months':    12,
    'num_open_accounts':            1,
    'num_student_loans':            3,
    'student_loan_outstanding_balance': 45_000,
    'mortgage_outstanding_balance': 0,
    'requested_credit_limit':       3_000,
    'predicted_default_probability': 0.82,
    'employment_stability_score':   0.10,
    'income_stability_score':       0.10,
    'financial_health_score':       0.15,
    'combined_risk_score':          480,
}

transport = httpx.ASGITransport(app=app)

async with httpx.AsyncClient(transport=transport, base_url='http://test') as client:

    # ── Test 1: Strong applicant should be approved ───────────────────────────
    r = await client.post('/predict', json=APPROVED_APPLICANT)
    assert r.status_code == 200, r.text
    result = r.json()
    print(f"[PASS] Strong applicant:  {result['decision']}  P={result['probability']:.4f}")

    # ── Test 2: High-risk applicant should be declined ────────────────────────
    r = await client.post('/predict', json=DECLINED_APPLICANT)
    assert r.status_code == 200, r.text
    result = r.json()
    print(f"[PASS] High-risk applicant: {result['decision']}  P={result['probability']:.4f}")

    # ── Test 3: Invalid employment_status → 422 Validation Error ─────────────
    bad = dict(APPROVED_APPLICANT, employment_status='InvalidStatus')
    r = await client.post('/predict', json=bad)
    assert r.status_code == 422
    print(f"[PASS] Invalid enum value → {r.status_code} Unprocessable Entity")

    # ── Test 4: Missing required field → 422 ─────────────────────────────────
    incomplete = {k: v for k, v in APPROVED_APPLICANT.items() if k != 'fico_score'}
    r = await client.post('/predict', json=incomplete)
    assert r.status_code == 422
    print(f"[PASS] Missing required field → {r.status_code} Unprocessable Entity")

INFO:httpx:HTTP Request: POST http://test/predict "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://test/predict "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://test/predict "HTTP/1.1 422 Unprocessable Entity"
INFO:httpx:HTTP Request: POST http://test/predict "HTTP/1.1 422 Unprocessable Entity"


[PASS] Strong applicant:  APPROVED  P=0.9995
[PASS] High-risk applicant: DECLINED  P=0.0338
[PASS] Invalid enum value → 422 Unprocessable Entity
[PASS] Missing required field → 422 Unprocessable Entity


In [34]:
server_should_exit = True  # signal the server to shut down after the tests

---
## Summary

| Requirement | File | Key detail |
|---|---|---|
| Serve `xgb_default.pkl` | `main.py` | `joblib.load` once at startup; `predict_proba` per request |
| Persist every decision | `models.py` + `database.py` | SQLite via SQLAlchemy ORM, `data/predictions.db` |
| Accept human-readable inputs | `schemas.py` → `ApplicationRequest` | 28 typed fields with Literal enums for dropdowns |
| Derive all 78 model features | `feature_engineering.py` | log1p, ordinal, one-hot, interaction, WoE |
| WoE bin maps | `models/woe_maps.json` | Rebuilt from X_train + y_train; loaded once at startup |
| Error code contract | `schemas.py` | `APIError` envelope + `ERROR_CODES` at `GET /error-codes` |
| Rate limiting | `limiter.py` + `main.py` | slowapi, 10 pred/min on `POST /predict` |
| Browser front-end | `templates/index.html` | 4-tab form (Personal / Income / Credit / Risk Scores) |
| JSON + HTML from one route | `main.py` | `HX-Request` header switch |

### What users no longer need to know

- That `annual_income` is stored as `log(1 + 42280) = 10.65`
- How to compute interaction features like `fico_utilization_score`
- What Weight-of-Evidence encoding is or how to calculate it
- The exact names of one-hot columns (`employment_status_Part-Time`, etc.)
